In [100]:
####################################
#ENVIRONMENT SETUP

In [101]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

In [102]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [103]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [104]:
#IMPORT FUNCTIONS (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from FUNCTIONS_Variable_Calculation import *

In [105]:
####################################
#LOADING CLASSES

In [106]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="EntrainmentCalculation", dataName="MassFluxCalculation",
                                dtype='int32')

=== CM1 Data Summary ===
 Simulation #:   1
 Resolution:     1km
 Time step:      5min
 Vertical levels:34
 Parcels:        1e6
 Data file:      /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Model/cm1r20.3/run/MODEL_OUTPUT/Simulation_One/cm1out_1km_5min_34nz.nc
 Parcel file:    /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Model/cm1r20.3/run/MODEL_OUTPUT/Simulation_One/cm1out_pdata_1km_5min_1e6np.nc
 Time steps:     133

=== DataManager Summary ===
 inputDirectory #:   /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/TimeSplitModelData
 outputDirectory #:   /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/EntrainmentCalculation
 inputDataDirectory #:   /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/TimeSplitModelData/Simulation_1_1km_5min_34nz/ModelData
 inputParcelDi

In [107]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='3min':
            num_jobs=30
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

Running timesteps from 0:6 



In [108]:
####################################
#FUNCTIONS

In [109]:
def MakeDataDictionary(variableNames,t,printstatement=False):
    timeString = ModelData.timeStrings[t]
    # print(f"Getting data from {timeString}","\n")
    
    dataDictionary = {variableName: CallLagrangianArray(ModelData, DataManager, timeString, variableName=variableName, printstatement=printstatement) 
                      for variableName in variableNames}      
    return dataDictionary

parcelData=ModelData.OpenParcel()
def GetSpatialData(t,wType="eulerian"):    
    variableNames = ['Z', 'Y', 'X', 'W']
    dataDictionary = MakeDataDictionary(variableNames,t)
    Z,Y,X, W = (dataDictionary[k] for k in variableNames)
    if wType=="eulerian":  #this W is eulerian w sampled onto lagrangian space #this project uses this method
        return Z,Y,X, W
    # elif wType=="lagrangian":
    #     W2 = parcelData['w'].isel(time=t).data #using lagrangian W instead of eulerian-lagrangian converted #not using for this project
    #     return Z,Y,X, W2

def GetAData(t):  
    # variableNames = [f'{Processed_string}A_g', f'{Processed_string}A_c']
    variableNames = [f'A_g', f'A_c'] #don't use processed version above, since that was only for the entrainment calculations

    dataDictionary = MakeDataDictionary(variableNames,t, printstatement=False)
    A_g,A_c = (dataDictionary[k] for k in variableNames)
    return A_g,A_c

Opened dataset: /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Model/cm1r20.3/run/MODEL_OUTPUT/Simulation_One/cm1out_pdata_1km_5min_1e6np.nc


In [110]:
def CalculateMassFlux(t, Z,Y,X, W, A):
    """
    Compute 3D Lagrangian mass flux.

    Returns a 3D (z, y, x) array where each grid cell contains the sum of A_i * W_i
    for all parcels that fall into that (Z, Y, X) index.

    Parameters:
    - A: (t, p) Lagrangian binary array (e.g., parcel mask or weight)
    - W: (t, p) vertical velocity (or relevant scalar for flux)
    - Z: (t, p) Lagrangian z index array
    - Y: (t, p) Lagrangian y index array
    - X: (t, p) Lagrangian x index array

    Returns:
    - MassFlux: (z, y, x) array of accumulated mass flux
    """
    
    # Initialize time and vertical dimension arrays
    Nz = ModelData.Nzh; Ny = ModelData.Nyh; Nx = ModelData.Nxh
    
    # Initialize result array
    MassFlux = np.zeros((Nz, Ny, Nx),dtype="float32")

    # Calculate Lagrangian Mass Flux
    LagrangianMassFlux = A*W    

    # Use np.add.at to accumulate values in the result array
    np.add.at(MassFlux, (Z, Y, X), LagrangianMassFlux)

    return MassFlux

In [111]:
def RunCalculation(t, Z,Y,X, W, A_g,A_c): 

    MassFlux_g = CalculateMassFlux(t, Z,Y,X, W, A=A_g)
    MassFlux_c = CalculateMassFlux(t, Z,Y,X, W, A=A_c)

    outputDictionary_MassFlux = {
        f"MassFlux_g": MassFlux_g,
        f"MassFlux_c": MassFlux_c
    }
        
    return outputDictionary_MassFlux

In [112]:
####################################
#CALCULATING MASS FLUX CONSTANT

In [113]:
#Functions

#constants
# def GetConstants():
#     Cp=1004 #Jkg-1K-1
#     Cv=717 #Jkg-1K-1
#     Rd=Cp-Cv #Jkg-1K-1
#     eps=0.608
#     return Cp,Cv,Rd,eps

def GetNumerics():
    Np=ModelData.Np #number of lagrangian parcles

    # Lx=(ModelData.xf[-1]-ModelData.xf[0])*1000 #x length (m)
    # Ly=(ModelData.yf[-1]-ModelData.yf[0])*1000 #y length (m)
    dt=ModelData.dt #seconds
    dy=ModelData.dy #meters
    dx=ModelData.dx #meters
    
    zfs=ModelData.zf*1000
    dz = np.diff(zfs)
    return Np, dt, dz,dy,dx

def zf(k):
    out=ModelData.zf[k]*1000
    return out

#calculation functions
# def rho(x,y,z,t):
#     p=data['prs'].isel(xh=x,yh=y,zh=z,time=t).item()
#     p0=101325 #Pa
#     theta=data['th'].isel(xh=x,yh=y,zh=z,time=t).item()
#     T=theta*(p/p0)**(Rd/Cp)
#     qv=data['qv'].isel(xh=x,yh=y,zh=z,time=t).item()
#     # Tv=T*(1+eps*qv)
#     Tv=T*(eps+qv)/(eps*(1+qv))
#     rho = p/(Rd*Tv)
#     out=rho
#     return out
    
def rho(x,y,z,rho_data_t):
    out=rho_data_t[z,y,x]
    return out
    
def Calculate_dm(t, dz,dy,dx, Np):
    timeString = ModelData.timeStrings[t]
    rho_data_t = CallVariable(ModelData, DataManager, timeString, "rho")
    
    #calculating
    m=0
    for k in range(ModelData.Nzh):
        dz=(zf(k+1)-zf(k))
        for j in range(ModelData.Nyh):
            for i in range(ModelData.Nxh):
                rho_out=rho(i,j,k,rho_data_t)
                m+=rho_out*dz

    #multiplying by integration differentials
    dm = m*dx*dy/Np
    return dm

def ComputeMassFluxConstant(t=0):
    #constants
    # [Cp,Cv,Rd,eps] = GetConstants()
    [Np, dt, dz,dy,dx] = GetNumerics()

    #calculation
    dm = Calculate_dm(t, dz,dy,dx, Np); print(dm)
    divisor=dz*dy*dx
    massFluxConstant = dm/divisor

    outputDictionary = {"massFluxConstant": massFluxConstant}
    return outputDictionary

In [114]:
try:
    DataManager_Entrainment = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="EntrainmentCalculation", dataName="MassFluxCalculation",dtype='int32',verbose=False)

    massFluxConstant = DataManager_Entrainment.LoadCalculations(DataManager_Entrainment.outputDataDirectory,
                                                                   dataName="MassFluxConstant",
                                                                   verbose=False)["massFluxConstant"]
except:
    #calculating
    outputDictionary_MassFluxConstant = ComputeMassFluxConstant()
    massFluxConstant = outputDictionary_MassFluxConstant["massFluxConstant"]
    
    #saving
    DataManager.SaveCalculations(DataManager.outputDataDirectory, outputDictionary_MassFluxConstant, dataName="MassFluxConstant",dtype="float32")
    
    #plotting
    plt.plot(massFluxConstant,ModelData.zh,color='black')
    plt.ylabel("z (km)")
    plt.xlabel("MassFlux Constant (kg/m^3)")
    plt.title("Plotting Vertical Profile of MassFlux Constant\n (due to stretched z-grid)");

def ApplyMassFluxConstant(outputDictionary_MassFlux):

    for key, value in outputDictionary_MassFlux.items():
        outputDictionary_MassFlux[key] = value * massFluxConstant[:, np.newaxis, np.newaxis]

In [115]:
##############################################
#RUNNING

In [ ]:
#running calculation
for t in tqdm(loop_elements, total=len(loop_elements)):
    
    # if np.mod(t,1)==0: print(f'Current time {t}')

    #loading data
    [Z,Y,X, W] = GetSpatialData(t)
    [A_g,A_c] = GetAData(t)

    #calculating variables
    outputDictionary_MassFlux = RunCalculation(t, Z,Y,X, W, A_g,A_c)

    #applying mass flux (density) constant
    ApplyMassFluxConstant(outputDictionary_MassFlux)
    
    #outputting
    timeString = ModelData.timeStrings[t]
    
    DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary_MassFlux, dataName=f"MassFlux",
                                   dtype="float32")
    

In [ ]:
##########################
#TESTING

In [ ]:
# #Plotting 

# fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# # plot 1 (contour)
# string = "PROCESSED_MassFlux_c" if PROCESSING == True else "MassFlux_c"
# a = MassFluxConstant[:,None,None]*outputDictionary_MassFlux[string]
# b = np.mean(a, axis=1)

# contour = axs[0].contourf(ModelData.xh, ModelData.zh, b)
# fig.colorbar(contour, ax=axs[0], label="M (kg/m^2/s)")
# axs[0].set_xlabel("x (km)")
# axs[0].set_ylabel("z (km)")
# axs[0].set_ylim(0, 20)
# axs[0].set_title("Mass Flux Contour")

# # plot 2 (line)
# b = np.mean(a, axis=(1, 2))
# axs[1].plot(b, ModelData.zh)
# axs[1].set_xlabel("M (kg/m^2/s)")
# axs[1].set_ylabel("z (km)")
# axs[1].set_title("Vertical Profile")

# plt.tight_layout()
# plt.show()

In [ ]:
# # #*testing lagrangian W can be negative even in eulerian updrafts
# # t=101

# # [Z,Y,X, W] = GetSpatialData(t)
# # [A_g,A_c] = GetAData(t)

# # a=W[A_g==1]
# # # a[a<0]
# # # x=Z[(A_g==1)&(W<0)]

# # a1=Z[x]
# # a2=Z[x]
# c=a2-a1
# plt.hist(c)